In [9]:
# ==========================================
# CELL 1: Alignment & Preprocessing
# ==========================================
import os
import re
import glob
import pandas as pd
from sklearn.preprocessing import StandardScaler

BASE_DIR = r"D:\B. Disease Grading"
GROUNDTRUTHS_DIR = os.path.join(BASE_DIR, "2. Groundtruths")

train_feat_csv = os.path.join(BASE_DIR, "train_features.csv")
test_feat_csv = os.path.join(BASE_DIR, "test_features.csv")

# Load extracted feature CSVs
df_train_feats = pd.read_csv(train_feat_csv)
df_test_feats = pd.read_csv(test_feat_csv)

# Locate Label CSVs inside 2. Groundtruths
gt_files = glob.glob(os.path.join(GROUNDTRUTHS_DIR, "**", "*.csv"), recursive=True)

train_gt_path = [f for f in gt_files if 'train' in os.path.basename(f).lower()][0]
test_gt_path = [f for f in gt_files if 'test' in os.path.basename(f).lower()][0]

df_train_gt = pd.read_csv(train_gt_path)
df_test_gt = pd.read_csv(test_gt_path)

print("Loaded Training Groundtruth:", os.path.basename(train_gt_path))
print("Loaded Testing Groundtruth:", os.path.basename(test_gt_path))

# Dynamic Column Detection
train_img_col = [c for c in df_train_gt.columns if 'image' in c.lower() or 'name' in c.lower()][0]
train_grade_col = [c for c in df_train_gt.columns if 'grade' in c.lower() or 'dr' in c.lower() or 'label' in c.lower()][0]

test_img_col = [c for c in df_test_gt.columns if 'image' in c.lower() or 'name' in c.lower()][0]
test_grade_col = [c for c in df_test_gt.columns if 'grade' in c.lower() or 'dr' in c.lower() or 'label' in c.lower()][0]

def extract_numeric_id(val):
    match = re.search(r'\d+', str(val))
    return int(match.group()) if match else -1

# Extract Numeric IDs
df_train_feats['Image_ID_Num'] = df_train_feats['Image_Name'].apply(extract_numeric_id)
df_train_gt['Image_ID_Num'] = df_train_gt[train_img_col].apply(extract_numeric_id)

df_test_feats['Image_ID_Num'] = df_test_feats['Image_Name'].apply(extract_numeric_id)
df_test_gt['Image_ID_Num'] = df_test_gt[test_img_col].apply(extract_numeric_id)

# Merge
df_train = pd.merge(df_train_feats, df_train_gt[['Image_ID_Num', train_grade_col]], on='Image_ID_Num', how='inner')
df_test = pd.merge(df_test_feats, df_test_gt[['Image_ID_Num', test_grade_col]], on='Image_ID_Num', how='inner')

feature_cols = ['Mean', 'StdDev', 'Variance', 'Contrast', 'Dissimilarity', 'Homogeneity', 'Energy', 'Correlation', 'LesionArea']

X_train = df_train[feature_cols]
y_train = df_train[train_grade_col]

X_test = df_test[feature_cols]
y_test = df_test[test_grade_col]

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nFinal Split: {X_train_scaled.shape[0]} Training Samples, {X_test_scaled.shape[0]} Testing Samples.")

Loaded Training Groundtruth: a. IDRiD_Disease Grading_Training Labels.csv
Loaded Testing Groundtruth: b. IDRiD_Disease Grading_Testing Labels.csv

Final Split: 413 Training Samples, 103 Testing Samples.
